# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset: {}".format(metadata.name))
print("Description: {}".format(metadata.description))

## 2. Data Overview
Review available record sets, fields, columns, and collect their `@id`s for further exploration.

Below we will print out the main record sets present, and for each, show its fields (with their `@id`s).

**Note:** All entities are referenced by their `@id` according to the FAIR² Croissant schema.

In [ ]:
# List available record sets and associated fields/columns by their @id
record_sets = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
if not record_sets:
    print("No record sets defined in the metadata. Attempting to enumerate them from Croissant schema...")
    # Try to extract from distribution/hasPart pattern (common in Croissant/FAIR2 datasets)
    main_record_sets = []
    for dist in metadata.to_json().get('distribution', []):
        part = dist.get('hasPart', [])
        if part:
            if isinstance(part, list):
                for p in part:
                    if isinstance(p, dict):
                        main_record_sets.append(p['@id'])
                    else:
                        main_record_sets.append(p)
            elif isinstance(part, dict):
                main_record_sets.append(part['@id'])
        else:
            # Sometimes the distribution itself is the record set
            main_record_sets.append(dist['@id'])
    record_sets = main_record_sets

if not record_sets:
    # Fallback: try known IDs for sen.science datasets
    # Based on knowledge cutoff, this dataset's main table is:
    # Typically, mass spec tables are called 'ProteinDataTable'. Let's use a known typical @id.
record_sets = ['https://api.app.sen.science/frontiers/7154140/4ee39b6d-b8d6-4e52-b497-8835d0f11e98']  # (Replace if you find the correct @id)

print("Record Sets (@id):")
for idx, rsid in enumerate(record_sets):
    print(f"{idx+1}. {rsid}")

# Introspect fields/columns for each record set
for record_set_id in record_sets:
    print(f"\nFields for Record Set {record_set_id}:")
    try:
        recs = list(dataset.records(record_set=record_set_id, max_records=1))
        if len(recs) > 0:
            print(list(recs[0].keys()))
        else:
            print("No fields found. The record set may not be accessible or may be a metadata object.")
    except Exception as e:
        print(f"Could not fetch records for {record_set_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set into pandas DataFrames
dataframes = {}
for record_set_id in record_sets:
    print(f"\nLoading data for record set {record_set_id} ...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            print(f"No records for {record_set_id}")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}")
        print(f"Fields/columns: {list(df.columns)}")
    except Exception as e:
        print(f"Error loading records from {record_set_id}: {e}")

# For further analysis, pick the main table (edit this @id to the main protein data table if known)
main_record_set_id = record_sets[0]

print(f"\nExample rows from main table {main_record_set_id}:")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, visualizing distributions or relationships, and grouping data by key attributes.

### Example: Filter proteins by abundance, normalize them, and group by a category

Replace `<numeric_field_id>` and `<group_field_id>` with the actual IDs or column names from your dataset, as displayed in the previous cell.

In [ ]:
# Choose a suitable numeric field and a group field based on data introspection

# --- These field names are typical for mass spec tables, adjust if different in your dataset ---
numeric_field = 'Total Abundance'  # e.g., could be 'Abundance', 'NormalizedAbundance', or similar
group_field = 'Sample'  # e.g., could be a sample condition, 'Condition', 'Treatment', etc.

df = dataframes[main_record_set_id]
if numeric_field not in df.columns:
    print(f"Field '{numeric_field}' not found, available fields are: {df.columns.tolist()}")
else:
    threshold = df[numeric_field].quantile(0.90)  # Filter to top 10% of abundance
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered to records with {numeric_field} > {threshold} (top 10% abundance): {len(filtered_df)} records")

    # Normalize
    filtered_df[f'{numeric_field}_normalized'] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )

    print(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print("\nGrouped data by {}:".format(group_field))
        print(grouped_df.head())
    else:
        print(f"Group field '{group_field}' not present in dataframe.")

## 5. Visualization
Visualize numeric distributions and group means for protein abundance or other interesting features. Modify field names as needed for your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the abundance field if available
if numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouped_df is defined (see EDA cell above), plot group means
    if 'grouped_df' in locals():
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- The notebook demonstrated programmatic access to a Croissant-based mass spectrometry dataset using `mlcroissant`.
- Main record sets and fields can be explored dynamically via `@id` references (essential for reproducible workflows).
- Routine EDA and pre-processing steps (filtering, normalization, grouping) as well as visualizations can be carried out seamlessly with pandas and seaborn/matplotlib.
- For domain-specific insights, refer to the detailed Croissant schema documentation for field semantics.

**Next Steps:** Try customizing the field names for your scientific questions or extend the EDA with statistical testing, deeper subsetting, or advanced visualizations.